In [5]:
import sqlite3
from langgraph.graph import StateGraph, START, END 
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command


llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db",
    check_same_thread = False,
)

config = {
    "configurable":{
        "thread_id":"2",
    },
}


In [6]:
class State(MessagesState):
    pass

graph_builder = StateGraph(State)



In [7]:
def chatbot(state:State):
    response = llm.invoke(state["messages"])
    return {
        "messages":[response]
    }

In [8]:
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

graph = graph_builder.compile(
    checkpointer = SqliteSaver(conn),
)

In [9]:
result = graph.invoke({
    "messages" :[
        {"role":"user","content":"Hello, how are you ? "}
    ]
    },
    config = config,

)

In [10]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello my name is Hyeseon
================================== Ai Message ==================================

Hello Hyeseon! How can I assist you today?
================================ Human Message =================================

Hello, how are you ? 
================================== Ai Message ==================================

Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How about you? How are you doing?
